In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import (
    shapiro,
    normaltest,
    ttest_ind,
    mannwhitneyu,
    chi2_contingency,
    f_oneway,
    kruskal,
    pointbiserialr
)

import statsmodels.api as sm
from statsmodels.stats.proportion import proportion_confint

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [4]:
# Generate  healthcare dataset


n = 3000

age = np.random.normal(42, 13, n).astype(int)
age = np.clip(age, 18, 80)

bmi = np.random.normal(30, 6, n)
bmi = np.clip(bmi, 16, 55)

glucose = np.random.normal(120, 32, n)
glucose = np.clip(glucose, 60, 260)

blood_pressure = np.random.normal(75, 12, n)
blood_pressure = np.clip(blood_pressure, 45, 130)

insulin = np.random.lognormal(mean=4.6, sigma=0.7, size=n)
insulin = np.clip(insulin, 10, 900)

cholesterol = np.random.normal(200, 38, n)
cholesterol = np.clip(cholesterol, 110, 380)

exercise_hours = np.random.normal(3.5, 2.0, n)
exercise_hours = np.clip(exercise_hours, 0, 12)

family_history = np.random.binomial(1, 0.32, n)

# Logistic formula for disease probability
logit = (
    -9
    + 0.035 * age
    + 0.055 * bmi
    + 0.032 * glucose
    + 0.015 * blood_pressure
    + 0.004 * cholesterol
    + 0.85 * family_history
    - 0.20 * exercise_hours
)

probability = 1 / (1 + np.exp(-logit))

outcome = np.random.binomial(1, probability)

health_df = pd.DataFrame({
    "Age": age,
    "BMI": np.round(bmi, 2),
    "Glucose": np.round(glucose, 2),
    "BloodPressure": np.round(blood_pressure, 2),
    "Insulin": np.round(insulin, 2),
    "Cholesterol": np.round(cholesterol, 2),
    "ExerciseHours": np.round(exercise_hours, 2),
    "FamilyHistory": family_history,
    "Disease": outcome
})

print("Dataset shape:", health_df.shape)
print(health_df.head())

Dataset shape: (3000, 9)
   Age    BMI  Glucose  BloodPressure  Insulin  Cholesterol  ExerciseHours  \
0   24  22.44    75.54          62.90   101.51       245.82           2.94   
1   41  31.53    89.77          74.72   117.65       175.87           1.90   
2   51  30.86   125.54          66.23   260.65       110.00           3.61   
3   45  25.88   179.36          85.49    33.49       214.05           3.82   
4   54  34.68   156.86          66.07   151.40       171.07           2.96   

   FamilyHistory  Disease  
0              0        0  
1              0        0  
2              1        1  
3              0        1  
4              1        1  


In [5]:
# Data overview


print("\nData Info:")
print(health_df.info())

print("\nMissing Values:")
print(health_df.isnull().sum())

print("\nDuplicate Rows:")
print(health_df.duplicated().sum())

print("\nBasic Description:")
print(health_df.describe())



Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Age            3000 non-null   int64  
 1   BMI            3000 non-null   float64
 2   Glucose        3000 non-null   float64
 3   BloodPressure  3000 non-null   float64
 4   Insulin        3000 non-null   float64
 5   Cholesterol    3000 non-null   float64
 6   ExerciseHours  3000 non-null   float64
 7   FamilyHistory  3000 non-null   int32  
 8   Disease        3000 non-null   int32  
dtypes: float64(6), int32(2), int64(1)
memory usage: 187.6 KB
None

Missing Values:
Age              0
BMI              0
Glucose          0
BloodPressure    0
Insulin          0
Cholesterol      0
ExerciseHours    0
FamilyHistory    0
Disease          0
dtype: int64

Duplicate Rows:
0

Basic Description:
               Age          BMI      Glucose  BloodPressure      Insulin  \
count  3000.000

In [6]:
# Descriptive statistics


numeric_cols = [
    "Age",
    "BMI",
    "Glucose",
    "BloodPressure",
    "Insulin",
    "Cholesterol",
    "ExerciseHours"
]

desc_stats = health_df[numeric_cols].agg([
    "mean",
    "median",
    "std",
    "var",
    "min",
    "max",
    "skew",
    "kurt"
]).T

desc_stats["range"] = health_df[numeric_cols].max() - health_df[numeric_cols].min()
desc_stats["IQR"] = health_df[numeric_cols].quantile(0.75) - health_df[numeric_cols].quantile(0.25)

print("\nDescriptive Statistics:")
print(desc_stats)




Descriptive Statistics:
                     mean   median        std          var    min     max  \
Age             41.856667   42.000  12.485088   155.877415   18.0   80.00   
BMI             29.986613   29.965   6.112496    37.362609   16.0   51.79   
Glucose        120.208920  118.975  31.606222   998.953252   60.0  237.45   
BloodPressure   75.281720   75.485  11.902691   141.674057   45.0  121.06   
Insulin        124.275470   99.890  94.225596  8878.462868   10.0  900.00   
Cholesterol    199.573613  200.465  37.782276  1427.500403  110.0  334.95   
ExerciseHours    3.574080    3.515   1.939232     3.760622    0.0   10.76   

                   skew       kurt   range      IQR  
Age            0.134601  -0.338935   62.00  17.0000  
BMI            0.070102  -0.209476   35.79   8.4100  
Glucose        0.199156  -0.299262  177.45  44.3575  
BloodPressure  0.024388  -0.040140   76.06  16.4100  
Insulin        2.567931  11.383749  890.00  95.1025  
Cholesterol   -0.030329  -0.172515